# Книга Excel: `read_excel` на своём компьютере

Блокнот к уроку «Книга Excel: листы, шапка, итоги» главы 3. В браузере курса те же примеры работают прямо в уроке; здесь — те же шаги в своём Jupyter на книге `zerno_finance.xlsx`, плюс запись результата обратно в Excel.

**Что нужно.** Библиотека openpyxl: в папке проекта из урока про uv выполните в терминале `uv add openpyxl` и перезапустите ядро (Kernel → Restart Kernel). Блокнот ищет книгу в папке `data` курса (`../../data`), а если не нашёл — рядом с собой.

**Как работать.** Выполняйте ячейки сверху вниз. В конце — три упражнения: место для кода помечено ✎, под каждым — ячейка самопроверки.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../../data")                   # блокнот лежит в notebooks/extra, данные — в data
if not (DATA / "zerno_finance.xlsx").exists():
    DATA = Path(".")                        # книга скачана рядом с блокнотом
BOOK = DATA / "zerno_finance.xlsx"
MONTHS = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен"]

try:
    import openpyxl
    print("pandas", pd.__version__, "| openpyxl", openpyxl.__version__, "| книга:", BOOK)
except ImportError:
    print("Нет openpyxl. Выполните в терминале: uv add openpyxl — и перезапустите ядро.")

## 1. Какие листы в книге

`sheet_name=None` читает все листы сразу и возвращает словарь «имя листа → таблица». Удобно, чтобы осмотреться.

In [ ]:
book = pd.read_excel(BOOK, sheet_name=None)
{name: df.shape for name, df in book.items()}

## 2. Лист «План»: пропустить шапку

Сначала — как есть. Заголовок листа стал названием первого столбца, остальные — `Unnamed`.

In [ ]:
raw = pd.read_excel(BOOK, sheet_name="План")
raw.head(4)

In [ ]:
plan = pd.read_excel(BOOK, sheet_name="План", skiprows=3)
print(plan.dtypes.value_counts())
plan

Названия столбцов на месте, месяцы — числа. В отличие от CSV, `decimal=","` не нужен: Excel хранит числа как числа, запятая — только способ их показать.

## 3. Лист «Факт»: шапка выше, столбцы буквами

У «Факта» четыре строки над таблицей. Столбцы выбираем буквами, как при выделении в Excel: A — код, D:L — январь–сентябрь.

In [ ]:
fact = pd.read_excel(BOOK, sheet_name="Факт", skiprows=4, usecols="A,D:L")
fact

## 4. Строка «Итого»: убрать и сверить

Сумма по кофейням должна совпасть с итоговой строкой. `assert` остановит блокнот, если это не так.

In [ ]:
body = fact[fact["Код"] != "Итого"]
total = fact[fact["Код"] == "Итого"]
by_shops = round(body[MONTHS].sum().sum(), 1)
from_total = round(total[MONTHS].sum().sum(), 1)
assert abs(by_shops - from_total) < 0.5, "сумма по кофейням не сходится с «Итого»"
print("факт за 9 месяцев:", by_shops, "тыс. ₽ — сходится с «Итого»")

## 5. План против факта

In [ ]:
p = plan[plan["Код"] != "Итого"].set_index("Код")[MONTHS].sum(axis=1)
f = body.set_index("Код")[MONTHS].sum(axis=1)
done = (f / p * 100).round(1)[p > 0].sort_values()
done

## 6. Сохранить отчёт обратно в Excel

`pd.ExcelWriter` пишет несколько таблиц на разные листы одной книги. Файл появится рядом с блокнотом.

In [ ]:
report = pd.DataFrame({"план": p, "факт": f, "выполнение, %": (f / p * 100).round(1)})[p > 0]
with pd.ExcelWriter("plan_fact_report.xlsx") as writer:
    report.to_excel(writer, sheet_name="Выполнение")
    body.to_excel(writer, sheet_name="Факт без итога", index=False)
list(pd.read_excel("plan_fact_report.xlsx", sheet_name=None))

## Упражнения

**1.** Прочитайте лист «Факт» не через `skiprows`, а через `header=` — так, чтобы получилась та же таблица `fact` (те же столбцы A, D:L). Результат — в `fact_h`.

In [ ]:
fact_h = None   # ✎ ваш код

In [ ]:
if fact_h is None:
    print("✎ упражнение 1 ещё не решено")
else:
    assert fact_h.equals(fact), "таблицы различаются: проверьте номер строки заголовка (считается с нуля)"
    print("✓ упражнение 1")

**2.** Прочитайте **все** листы книги одним вызовом и сохраните в `sheets` список их имён в том порядке, в каком они идут в книге.

In [ ]:
sheets = None   # ✎ ваш код

In [ ]:
if sheets is None:
    print("✎ упражнение 2 ещё не решено")
else:
    assert list(sheets) == ["Описание", "План", "Факт"], f"получилось {sheets}"
    print("✓ упражнение 2")

**3.** В каких месяцах сеть целиком (сумма по кофейням) выполнила план меньше чем на 97 %? Сохраните в `weak` список названий месяцев («Янв», «Фев», …).

In [ ]:
weak = None   # ✎ ваш код

In [ ]:
if weak is None:
    print("✎ упражнение 3 ещё не решено")
else:
    pm = plan[plan["Код"] != "Итого"][MONTHS].sum()
    fm = body[MONTHS].sum()
    want = list((fm / pm * 100)[lambda s: s < 97].index)
    assert list(weak) == want, f"получилось {list(weak)}"
    print("✓ упражнение 3:", want)